In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# ESG Portfolio (EU) — Fixed Strategy\n",
    "\n",
    "**Updated for ESSCA Simulation 2026**\n",
    "\n",
    "### Major Updates:\n",
    "1. **Screener:** ESG Score >= 50 (Best-in-class filter)\n",
    "2. **Strategy:** Smart Momentum (Top Momentum stocks + Inverse Volatility Weighting) -> Target Max Sharpe Ratio\n",
    "3. **Dashboard:** Added visual analytics for Portfolio Weights, ESG Score, and Risk Metrics.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from IPython.display import display\n",
    "\n",
    "import refinitiv.data as rd\n",
    "\n",
    "pd.set_option(\"display.max_rows\", 50)\n",
    "pd.set_option(\"display.max_columns\", 50)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 0) Config\n",
    "Set `history_end` to TODAY (2026-02-19) for the first trading day."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "CFG = {\n",
    "  \"top_n\": 300,\n",
    "  \"markets\": [\"XPAR\", \"XAMS\", \"XBRU\", \"XMAD\"],\n",
    "  \"history_start\": \"2025-01-01\",\n",
    "  \"history_end\": \"2026-02-19\" \n",
    "}\n",
    "CFG"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1) Open Refinitiv session"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "try:\n",
    "    rd.open_session()\n",
    "    print(\"✅ Session opened\")\n",
    "except Exception as e:\n",
    "    print(f\"⚠️ Error opening session: {e}\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2) Screener: EU stocks with ESG Score >= 50 (FIXED)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def _as_df(x):\n",
    "    \"\"\"Refinitiv sometimes returns a DataFrame directly, or a response object with .data.df\"\"\"\n",
    "    if isinstance(x, pd.DataFrame):\n",
    "        return x\n",
    "    if hasattr(x, \"data\") and hasattr(x.data, \"df\"):\n",
    "        return x.data.df\n",
    "    return x\n",
    "\n",
    "def get_top_esg_stocks(cfg):\n",
    "    mic_codes = \", \".join([f\"'{m}'\" for m in cfg[\"markets\"]])\n",
    "\n",
    "    # --- UPDATED QUERY: ESG >= 50 ---\n",
    "    screener_query = (\n",
    "        \"SCREEN(\"\n",
    "        \"U(IN(Equity(active,public,primary))), \"\n",
    "        f\"IN(TR.ExchangeMarketIdCode, {mic_codes}), \"\n",
    "        \"TR.TRESGScore >= 50\" \n",
    "        \")\"\n",
    "    )\n",
    "\n",
    "    fields = [\n",
    "        \"TR.CommonName\",\n",
    "        \"TR.ISIN\",\n",
    "        \"TR.ExchangeName\",\n",
    "        \"TR.ExchangeMarketIdCode\",\n",
    "        \"TR.Currency\",\n",
    "        \"TR.TRESGScore\",\n",
    "        \"TR.CompanyMarketCap\",\n",
    "        \"TR.PriceClose\",\n",
    "    ]\n",
    "\n",
    "    print(f\"Searching with query: {screener_query}...\")\n",
    "    try:\n",
    "        raw = rd.get_data(universe=screener_query, fields=fields)\n",
    "        df = _as_df(raw).copy()\n",
    "    except Exception as e:\n",
    "        print(f\"Error fetching screener data: {e}\")\n",
    "        return pd.DataFrame()\n",
    "\n",
    "    if \"Instrument\" in df.columns and \"RIC\" not in df.columns:\n",
    "        df = df.rename(columns={\"Instrument\": \"RIC\"})\n",
    "\n",
    "    # Map columns\n",
    "    rename_map = {\n",
    "        \"Company Common Name\": \"TR.CommonName\",\n",
    "        \"Company Market Cap\": \"TR.CompanyMarketCap\",\n",
    "        \"ESG Score\": \"TR.TRESGScore\",\n",
    "        \"Price Close\": \"TR.PriceClose\",\n",
    "        \"Exchange Name\": \"TR.ExchangeName\",\n",
    "        \"Exchange Market Identifier Code\": \"TR.ExchangeMarketIdCode\",\n",
    "    }\n",
    "    for k, v in rename_map.items():\n",
    "        if k in df.columns and v not in df.columns:\n",
    "            df = df.rename(columns={k: v})\n",
    "\n",
    "    if \"TR.Currency\" in df.columns:\n",
    "        df = df[df[\"TR.Currency\"].eq(\"EUR\")].copy()\n",
    "\n",
    "    top_n = int(cfg.get(\"top_n\", 300))\n",
    "    if \"TR.CompanyMarketCap\" in df.columns:\n",
    "        # Sort by Market Cap to ensure liquidity\n",
    "        df = df.sort_values(\"TR.CompanyMarketCap\", ascending=False)\n",
    "    df = df.head(top_n).reset_index(drop=True)\n",
    "\n",
    "    return df\n",
    "\n",
    "df_top = get_top_esg_stocks(CFG)\n",
    "print(f\"Stocks found: {len(df_top)}\")\n",
    "df_top.head(10)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Reduce to a stable investable set (Top 60 by Market Cap)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def pick_investable(df_top, n_mcap=60):\n",
    "    df = df_top.copy()\n",
    "    if \"RIC\" not in df.columns:\n",
    "        raise KeyError(\"Missing 'RIC' column. Check df_top.columns\")\n",
    "\n",
    "    if \"TR.CompanyMarketCap\" in df.columns:\n",
    "        df = df.dropna(subset=[\"RIC\", \"TR.CompanyMarketCap\"]).sort_values(\"TR.CompanyMarketCap\", ascending=False)\n",
    "        rics = df.head(n_mcap)[\"RIC\"].astype(str).tolist()\n",
    "    else:\n",
    "        rics = df[\"RIC\"].dropna().astype(str).unique().tolist()[:n_mcap]\n",
    "    return rics\n",
    "\n",
    "rics60 = pick_investable(df_top, n_mcap=60)\n",
    "print(f\"Selected {len(rics60)} tickers for universe.\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3) Download daily prices and build price matrix"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def get_price_matrix(rics, start, end):\n",
    "    print(f\"Fetching prices from {start} to {end}...\")\n",
    "    try:\n",
    "        raw = rd.get_history(\n",
    "            universe=rics,\n",
    "            fields=[\"TR.PriceClose\"],\n",
    "            start=start,\n",
    "            end=end,\n",
    "            interval=\"daily\"\n",
    "        )\n",
    "        df = _as_df(raw).copy()\n",
    "\n",
    "        if \"Date\" in df.columns and \"Instrument\" in df.columns:\n",
    "            price_col = \"TR.PriceClose\" if \"TR.PriceClose\" in df.columns else (\"Price Close\" if \"Price Close\" in df.columns else None)\n",
    "            if price_col is None:\n",
    "                raise KeyError(f\"Can't find price column in history: {df.columns.tolist()}\")\n",
    "            px = df.pivot(index=\"Date\", columns=\"Instrument\", values=price_col).sort_index()\n",
    "        else:\n",
    "            px = df.copy()\n",
    "\n",
    "        px = px.dropna(axis=1, how=\"any\")\n",
    "        return px\n",
    "    except Exception as e:\n",
    "        print(f\"Error fetching prices: {e}\")\n",
    "        return pd.DataFrame()\n",
    "\n",
    "px = get_price_matrix(rics60, CFG[\"history_start\"], CFG[\"history_end\"])\n",
    "print(\"Price Matrix Shape:\", px.shape)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4) Returns (clean)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "rets = px.pct_change(fill_method=None).dropna(how=\"all\")\n",
    "rets = rets.dropna(axis=1, how=\"any\")\n",
    "print(\"Returns shape:\", rets.shape)\n",
    "rets.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5) Better Strategy: Smart Momentum (Momentum + Inverse Volatility)\n",
    "\n",
    "This strategy selects the highest momentum stocks but weights them by **Inverse Volatility**.\n",
    "- **Why?** It captures upside trends (Momentum) while reducing portfolio risk (Inverse Vol) to maximize the **Sharpe Ratio**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def build_smart_momentum_portfolio(rets_df, n=20, lookback=15):\n",
    "    \"\"\"\n",
    "    Selects top N momentum stocks (15-day lookback).\n",
    "    Weights them using Inverse Volatility (Low risk = Higher weight).\n",
    "    \"\"\"\n",
    "    # 1. Calculate Momentum (Cumulative return over lookback period)\n",
    "    mom = (1 + rets_df.tail(lookback)).prod() - 1\n",
    "    mom = mom.dropna()\n",
    "    \n",
    "    # Pick Top N assets\n",
    "    top_assets = mom.sort_values(ascending=False).head(n).index\n",
    "    \n",
    "    # 2. Calculate Volatility (Annualized Std Dev of last 30 days)\n",
    "    recent_rets = rets_df[top_assets].tail(30)\n",
    "    vol = recent_rets.std() * np.sqrt(252)\n",
    "    vol = vol.replace(0, 0.0001) # Avoid div by zero\n",
    "    \n",
    "    # 3. Inverse Volatility Weighting\n",
    "    inv_vol = 1.0 / vol\n",
    "    weights = inv_vol / inv_vol.sum()\n",
    "    \n",
    "    # 4. Cap max weight at 10% to ensure diversification\n",
    "    weights = weights.clip(upper=0.10)\n",
    "    weights = weights / weights.sum() # Normalize again\n",
    "    \n",
    "    return weights.sort_values(ascending=False), mom[top_assets]\n",
    "\n",
    "# EXECUTE STRATEGY\n",
    "w_smart, mom_smart = build_smart_momentum_portfolio(rets, n=20, lookback=15)\n",
    "\n",
    "print(\"Top 10 Holdings:\")\n",
    "print(w_smart.head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6) Portfolio Dashboard & Visualization (NEW)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_portfolio_dashboard(weights_series, rets_df, screener_df):\n",
    "    \"\"\"\n",
    "    Generates a visual dashboard for the portfolio.\n",
    "    \"\"\"\n",
    "    port_assets = weights_series.index.tolist()\n",
    "    \n",
    "    # 1. Get ESG Scores\n",
    "    esg_mapping = screener_df.set_index('RIC')['TR.TRESGScore'].to_dict()\n",
    "    \n",
    "    # 2. Calculate Metrics\n",
    "    ann_ret_assets = rets_df[port_assets].mean() * 252\n",
    "    ann_vol_assets = rets_df[port_assets].std() * np.sqrt(252)\n",
    "\n",
    "    summary_df = pd.DataFrame({\n",
    "        'Weight': weights_series,\n",
    "        'ESG_Score': weights_series.index.map(lambda x: esg_mapping.get(x, np.nan)),\n",
    "        'Exp_Ann_Return': ann_ret_assets,\n",
    "        'Ann_Volatility': ann_vol_assets\n",
    "    })\n",
    "    \n",
    "    # Portfolio Level Metrics\n",
    "    port_return = (summary_df['Weight'] * summary_df['Exp_Ann_Return']).sum()\n",
    "    cov_matrix = rets_df[port_assets].cov() * 252\n",
    "    port_vol = np.sqrt(np.dot(weights_series.values.T, np.dot(cov_matrix.values, weights_series.values)))\n",
    "    port_sharpe = port_return / port_vol if port_vol > 0 else 0\n",
    "    port_esg = (summary_df['Weight'] * summary_df['ESG_Score']).sum()\n",
    "\n",
    "    # --- DISPLAY METRICS ---\n",
    "    print(\"=\"*40)\n",
    "    print(\"🏆 PORTFOLIO OVERVIEW (Estimated) 🏆\")\n",
    "    print(\"=\"*40)\n",
    "    print(f\"🔹 Exp. Ann. Return   : {port_return:.2%}\")\n",
    "    print(f\"🔹 Ann. Volatility    : {port_vol:.2%}\")\n",
    "    print(f\"🔹 Sharpe Ratio       : {port_sharpe:.2f}\")\n",
    "    print(f\"🔹 Weighted ESG Score : {port_esg:.2f} / 100\")\n",
    "    print(\"=\"*40)\n",
    "    \n",
    "    # --- DISPLAY TABLE ---\n",
    "    print(\"\\n📊 ASSET ALLOCATION DETAILED VIEW:\")\n",
    "    try:\n",
    "        styled_df = summary_df.sort_values('Weight', ascending=False).style.format({\n",
    "            'Weight': '{:.2%}',\n",
    "            'ESG_Score': '{:.1f}',\n",
    "            'Exp_Ann_Return': '{:.2%}',\n",
    "            'Ann_Volatility': '{:.2%}'\n",
    "        }).background_gradient(cmap='Greens', subset=['ESG_Score'])\n",
    "        display(styled_df)\n",
    "    except:\n",
    "        display(summary_df.sort_values('Weight', ascending=False))\n",
    "\n",
    "    # --- PLOTS ---\n",
    "    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))\n",
    "    \n",
    "    # Pie Chart\n",
    "    w_plot = summary_df[summary_df['Weight'] > 0.01]\n",
    "    ax1.pie(w_plot['Weight'], labels=w_plot.index, autopct='%1.1f%%', startangle=140, cmap='tab20')\n",
    "    ax1.set_title(\"Portfolio Weights (>1%)\", fontsize=14, fontweight='bold')\n",
    "    \n",
    "    # Bar Chart ESG\n",
    "    if not summary_df['ESG_Score'].isnull().all():\n",
    "        sns.barplot(x=summary_df.index, y=summary_df['ESG_Score'], ax=ax2, hue=summary_df.index, legend=False, palette='viridis')\n",
    "        ax2.axhline(port_esg, color='red', linestyle='--', label=f'Avg ESG ({port_esg:.1f})')\n",
    "        ax2.axhline(50, color='gray', linestyle=':', label='Min ESG (50)')\n",
    "        ax2.set_title(\"ESG Scores per Stock\", fontsize=14, fontweight='bold')\n",
    "        ax2.tick_params(axis='x', rotation=45)\n",
    "        ax2.legend()\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "# Run Dashboard\n",
    "generate_portfolio_dashboard(w_smart, rets, df_top)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7) Export for StockTrak (Trading)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "target = w_smart.copy()\n",
    "\n",
    "out = (target.rename(\"weight\")\n",
    "       .reset_index()\n",
    "       .rename(columns={\"index\":\"RIC\"})\n",
    "       .sort_values(\"weight\", ascending=False)\n",
    "       .reset_index(drop=True))\n",
    "\n",
    "out.to_csv(\"target_portfolio_smart.csv\", index=False)\n",
    "print(\"✅ Saved 'target_portfolio_smart.csv' successfully.\")\n",
    "out.head(20)\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined

In [ ]:
# out: DataFrame có cột RIC, weight
# df_top: DataFrame có cột RIC và ESG score (friendly name hoặc TR.TRESGScore)

# tìm đúng cột ESG score trong df_top
esg_col = "ESG Score" if "ESG Score" in df_top.columns else ("TR.TRESGScore" if "TR.TRESGScore" in df_top.columns else None)
if esg_col is None:
    raise KeyError("Không tìm thấy cột ESG Score trong df_top")

port_esg = out.merge(df_top[["RIC", esg_col]], onPrice Close weight 0 NTGY.MC 0.05 1 CLNX.MC 0.05 2 BBVA.MC 0.05 3 AIR.PA 0.05 4 SASY.PA 0.05 5 STMPA.PA 0.05 6 PHG.AS 0.05 7 ABI.BR 0.05 8 ENGIE.PA 0.05 9 VIE.PA 0.05 10 HEIN.AS 0.05 11 MT.AS 0.05 12 ORAN.PA 0.05 13 HEIO.AS 0.05 14 IBE.MC 0.05 15 TEF.MC 0.05 16 DIOR.PA 0.05 17 LVMH.PA 0.05 18 ASML.AS 0.05 19 SAN.MC 0.0="RIC", how="left")

port_esg.rename(columns={esg_col: "ESG_Score"}, inplace=True)
port_esg


KeyError: 'RIC'